# ETF Knowledge Graph — Exploration

Build a tuple-based knowledge graph from live ETF data, then visualise it as an
interactive network. No graph database required — just Python lists and dicts.

**Pipeline:** Scrape → Normalise → Build tuples → NetworkX graph → Interactive Pyvis HTML

In [ ]:
import html as html_mod
import re
from collections import defaultdict
from typing import Any

import httpx
import justetf_scraping
import networkx as nx
import pandas as pd
from pyvis.network import Network

## 1. Seed ETFs & scraping helpers

In [ ]:
SEED_ETFS = [
    {"isin": "IE00BGV5VN51", "ticker": "XAIX.L", "name": "Xtrackers AI & Big Data UCITS ETF 1C"},
    {"isin": "IE00BMC38736", "ticker": "SMGB.L", "name": "iShares MSCI Global Semiconductors UCITS ETF"},
    {"isin": "IE00BMH5Y327", "ticker": "VPNG.L", "name": "Global X Data Center REITs & Digital Infra UCITS ETF"},
    {"isin": "IE000NDWFGA5", "ticker": "URNG.L", "name": "Global X Uranium UCITS ETF"},
    {"isin": "IE00B3CNHG25", "ticker": "AUCP.L", "name": "L&G Gold Mining UCITS ETF"},
    {"isin": "IE00B4ND3602", "ticker": "SGLN.L", "name": "iShares Physical Gold ETC"},
    {"isin": "IE000JCW3DZ3", "ticker": "ARMG.L", "name": "Global X Defence Tech UCITS ETF"},
]

JUSTETF_PROFILE_URL = "https://www.justetf.com/en/etf-profile.html"
JUSTETF_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "text/html,application/xhtml+xml",
}

In [ ]:
def _safe_float(val: Any) -> float | None:
    if val is None:
        return None
    try:
        return float(val)
    except (ValueError, TypeError):
        return None


def _extract_top_holdings(page_html: str) -> list[dict]:
    names = re.findall(
        r'data-testid="tl_etf-holdings_top-holdings_link_name"[^>]*title="([^"]+)"',
        page_html,
    )
    pcts = re.findall(
        r'data-testid="tl_etf-holdings_top-holdings_value_percentage"[^>]*>([^<]+)<',
        page_html,
    )
    out = []
    for i, name in enumerate(names):
        weight = _safe_float(pcts[i].replace("%", "").strip()) if i < len(pcts) else None
        if weight is not None:
            weight /= 100
        out.append({"name": html_mod.unescape(name.strip()), "weight": weight})
    return out


def _extract_allocation_block(page_html: str, block_type: str) -> list[dict]:
    names = re.findall(
        rf'data-testid="tl_etf-holdings_{re.escape(block_type)}_value_name"[^>]*>([^<]+)<',
        page_html,
    )
    pcts = re.findall(
        rf'data-testid="tl_etf-holdings_{re.escape(block_type)}_value_percentage"[^>]*>([^<]+)<',
        page_html,
    )
    out = []
    for i, name in enumerate(names):
        pct = _safe_float(pcts[i].replace("%", "").strip()) if i < len(pcts) else None
        out.append({"name": html_mod.unescape(name.strip()), "percentage": pct})
    return out


async def scrape_profile(isin: str) -> dict:
    """Scrape the justETF profile page for metadata not in get_etf_overview."""
    meta: dict[str, Any] = {}
    async with httpx.AsyncClient(timeout=20.0, follow_redirects=True) as client:
        resp = await client.get(
            JUSTETF_PROFILE_URL,
            params={"isin": isin},
            headers=JUSTETF_HEADERS,
        )
        resp.raise_for_status()
        page = resp.text

    testid_map = {
        "fund-provider": "fund_provider",
        "legal-structure": "legal_structure",
        "strategy-risk": "strategy_risk",
        "sustainable": "sustainability",
        "fund-currency": "fund_currency",
        "currency-hedge": "currency_risk",
        "distribution-interval": "distribution_frequency",
        "investment-focus": "investment_focus",
        "index-name": "index_name",
        "ter": "ter_raw",
        "replication": "replication",
        "launch-date": "inception_date_raw",
        "distribution-policy": "distribution",
        "domicile-country": "domicile",
    }
    for testid, key in testid_map.items():
        pattern = re.compile(
            rf'data-testid="[^"]*_value_{re.escape(testid)}"[^>]*>([^<]+)<',
            re.IGNORECASE,
        )
        m = pattern.search(page)
        if m:
            val = html_mod.unescape(m.group(1).strip())
            if val and val != "-":
                meta[key] = val

    if "ter_raw" in meta:
        m = re.search(r"([\d.]+)%", meta.pop("ter_raw"))
        if m:
            meta["ter"] = float(m.group(1)) / 100

    meta["holdings"] = _extract_top_holdings(page)
    meta["countries"] = _extract_allocation_block(page, "countries")
    meta["sectors"] = _extract_allocation_block(page, "sectors")
    return meta

## 2. Fetch data for all 7 ETFs

Combines `justetf_scraping.get_etf_overview()` with the profile-page HTML scraper.
This may take 30-60 seconds — one HTTP request per ETF.

In [ ]:
raw_data: dict[str, dict] = {}

for etf in SEED_ETFS:
    isin = etf["isin"]
    print(f"Fetching {etf['ticker']} ({isin}) ...")

    overview = {}
    try:
        overview = justetf_scraping.get_etf_overview(isin)
    except Exception as e:
        print(f"  ⚠ get_etf_overview failed: {e}")

    profile = {}
    try:
        profile = await scrape_profile(isin)
    except Exception as e:
        print(f"  ⚠ profile scrape failed: {e}")

    raw_data[isin] = {
        "seed": etf,
        "overview": overview,
        "profile": profile,
    }
    print(f"  ✓ overview keys: {list(overview.keys())[:8]}...")
    print(f"  ✓ profile keys:  {list(profile.keys())[:8]}...")

print(f"\nDone — fetched data for {len(raw_data)} ETFs.")

## 3. Build the Knowledge Graph as tuples

Two stores:
- `nodes`: dict keyed by node ID → `{id, type, props}`
- `edges`: list of `(source_id, relation, target_id, props)` tuples

In [ ]:
nodes: dict[str, dict] = {}
edges: list[tuple[str, str, str, dict]] = []


def add_node(node_id: str, node_type: str, props: dict | None = None) -> str:
    if node_id not in nodes:
        nodes[node_id] = {"id": node_id, "type": node_type, "props": props or {}}
    else:
        nodes[node_id]["props"].update(props or {})
    return node_id


def add_edge(source: str, relation: str, target: str, props: dict | None = None):
    edges.append((source, relation, target, props or {}))


def slugify(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")

In [ ]:
holding_to_etfs: dict[str, list[str]] = defaultdict(list)

for isin, data in raw_data.items():
    seed = data["seed"]
    overview = data.get("overview") or {}
    profile = data.get("profile") or {}

    # --- ETF node ---
    etf_id = add_node(f"etf:{isin}", "ETF", {
        "name": seed["name"],
        "ticker": seed["ticker"],
        "isin": isin,
        "ter": overview.get("ter") or profile.get("ter"),
        "aum_eur": overview.get("fund_size_eur") or overview.get("fund_size"),
        "replication": overview.get("replication") or profile.get("replication"),
        "distribution": overview.get("distribution_policy") or profile.get("distribution"),
        "holdings_count": overview.get("number_of_holdings"),
        "investment_focus": profile.get("investment_focus"),
        "domicile": overview.get("fund_domicile") or profile.get("domicile"),
    })

    # --- Provider → ISSUED_BY ---
    provider_name = profile.get("fund_provider")
    if provider_name:
        pid = add_node(f"provider:{slugify(provider_name)}", "Provider", {"name": provider_name})
        add_edge(etf_id, "ISSUED_BY", pid)

    # --- Index → TRACKS ---
    index_name = overview.get("index") or profile.get("index_name")
    if index_name:
        idx_id = add_node(f"index:{slugify(index_name)}", "Index", {"name": index_name})
        add_edge(etf_id, "TRACKS", idx_id)

    # --- Exchange → LISTED_ON ---
    exchange = "LSE"
    exc_id = add_node(f"exchange:{slugify(exchange)}", "Exchange", {"name": exchange})
    add_edge(etf_id, "LISTED_ON", exc_id)

    # --- Holdings → HOLDS ---
    holdings = overview.get("top_holdings") or overview.get("holdings") or []
    if not holdings:
        holdings = profile.get("holdings") or []

    for h in holdings:
        h_name = h.get("name", "Unknown")
        h_isin = h.get("isin")
        h_key = h_isin if h_isin else slugify(h_name)
        h_id = add_node(f"holding:{h_key}", "Holding", {
            "name": h_name,
            "isin": h_isin,
            "ticker": h.get("ticker"),
        })
        weight = _safe_float(h.get("weight") or h.get("share"))
        add_edge(etf_id, "HOLDS", h_id, {"weight": weight})
        holding_to_etfs[h_id].append(etf_id)

    # --- Countries → EXPOSED_TO ---
    countries = overview.get("countries") or []
    if not countries:
        countries = profile.get("countries") or []
    for c in countries:
        c_name = c.get("name") or c.get("country") or ""
        if not c_name:
            continue
        c_id = add_node(f"country:{slugify(c_name)}", "Country", {"name": c_name})
        pct = _safe_float(c.get("share") or c.get("percentage"))
        add_edge(etf_id, "EXPOSED_TO", c_id, {"percentage": pct})

    # --- Sectors → INVESTS_IN ---
    sectors = overview.get("sectors") or []
    if not sectors:
        sectors = profile.get("sectors") or []
    for s in sectors:
        s_name = s.get("name") or s.get("sector") or ""
        if not s_name:
            continue
        s_id = add_node(f"sector:{slugify(s_name)}", "Sector", {"name": s_name})
        pct = _safe_float(s.get("share") or s.get("percentage"))
        add_edge(etf_id, "INVESTS_IN", s_id, {"percentage": pct})

print(f"Nodes: {len(nodes)}  |  Edges: {len(edges)}")
node_type_counts = defaultdict(int)
for n in nodes.values():
    node_type_counts[n["type"]] += 1
print("Node types:", dict(node_type_counts))

edge_type_counts = defaultdict(int)
for e in edges:
    edge_type_counts[e[1]] += 1
print("Edge types:", dict(edge_type_counts))

## 4. Compute OVERLAPS_WITH edges

Two ETFs overlap when they share at least one holding.

In [ ]:
etf_to_holdings: dict[str, set[str]] = defaultdict(set)
for h_id, etf_ids in holding_to_etfs.items():
    for eid in etf_ids:
        etf_to_holdings[eid].add(h_id)

etf_ids_sorted = sorted(etf_to_holdings.keys())
overlap_count = 0
for i, a in enumerate(etf_ids_sorted):
    for b in etf_ids_sorted[i + 1:]:
        shared = etf_to_holdings[a] & etf_to_holdings[b]
        if shared:
            add_edge(a, "OVERLAPS_WITH", b, {
                "shared_count": len(shared),
                "shared_holdings": sorted(shared),
            })
            overlap_count += 1
            a_name = nodes[a]["props"].get("ticker", a)
            b_name = nodes[b]["props"].get("ticker", b)
            print(f"  {a_name} ↔ {b_name}: {len(shared)} shared holdings")

print(f"\nAdded {overlap_count} OVERLAPS_WITH edges.")
print(f"Total graph: {len(nodes)} nodes, {len(edges)} edges")

## 5. Inspect the tuples

Preview the raw tuple data before visualisation.

In [ ]:
edges_df = pd.DataFrame(
    [(s, r, t, p) for s, r, t, p in edges],
    columns=["source", "relation", "target", "properties"],
)
print(f"Edge tuples: {len(edges_df)}")
print(f"\nRelation breakdown:\n{edges_df['relation'].value_counts().to_string()}")
print(f"\n--- Sample tuples ---")
edges_df.head(15)

In [ ]:
nodes_df = pd.DataFrame([
    {"id": n["id"], "type": n["type"], **n["props"]}
    for n in nodes.values()
])
print(f"Nodes: {len(nodes_df)}")
print(f"\nType breakdown:\n{nodes_df['type'].value_counts().to_string()}")
print(f"\n--- Sample nodes ---")
nodes_df.head(15)

## 6. Build NetworkX graph

In [ ]:
G = nx.MultiDiGraph()

for node_id, node in nodes.items():
    G.add_node(node_id, type=node["type"], **node["props"])

for src, rel, tgt, props in edges:
    G.add_edge(src, tgt, relation=rel, **props)

print(f"NetworkX graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Connected components (undirected): {nx.number_connected_components(G.to_undirected())}")

etf_nodes = [n for n, d in G.nodes(data=True) if d.get('type') == 'ETF']
print(f"\nDegree of ETF nodes (total connections):")
for n in etf_nodes:
    ticker = G.nodes[n].get('ticker', n)
    print(f"  {ticker}: {G.degree(n)} connections")

## 7. Interactive visualisation with Pyvis

Generates a standalone HTML file with physics-based layout.
Nodes are coloured and sized by type, edges are coloured by relation.

In [ ]:
NODE_STYLE = {
    "ETF":      {"color": "#FF6B6B", "size": 35, "shape": "dot"},
    "Provider": {"color": "#4ECDC4", "size": 25, "shape": "diamond"},
    "Holding":  {"color": "#45B7D1", "size": 12, "shape": "dot"},
    "Sector":   {"color": "#96CEB4", "size": 20, "shape": "triangle"},
    "Country":  {"color": "#FFEAA7", "size": 20, "shape": "square"},
    "Index":    {"color": "#DDA0DD", "size": 22, "shape": "star"},
    "Exchange": {"color": "#FFB347", "size": 20, "shape": "diamond"},
}

EDGE_COLORS = {
    "ISSUED_BY":     "#4ECDC4",
    "TRACKS":        "#DDA0DD",
    "LISTED_ON":     "#FFB347",
    "HOLDS":         "#45B7D1",
    "EXPOSED_TO":    "#FFEAA7",
    "INVESTS_IN":    "#96CEB4",
    "OVERLAPS_WITH": "#FF6B6B",
}


def build_label(node: dict) -> str:
    ntype = node["type"]
    props = node["props"]
    if ntype == "ETF":
        return props.get("ticker", props.get("name", node["id"]))
    return props.get("name", node["id"].split(":", 1)[-1])


def build_title(node: dict) -> str:
    """Hover tooltip — show all properties."""
    lines = [f"<b>{node['type']}</b>: {build_label(node)}"]
    for k, v in node["props"].items():
        if v is not None and k not in ("name",):
            lines.append(f"  {k}: {v}")
    return "<br>".join(lines)


def build_pyvis(output_file: str = "etf_knowledge_graph.html"):
    net = Network(
        height="800px",
        width="100%",
        directed=True,
        notebook=True,
        cdn_resources="remote",
    )
    net.barnes_hut(
        gravity=-3000,
        central_gravity=0.3,
        spring_length=150,
        spring_strength=0.04,
        damping=0.09,
    )

    for node_id, node in nodes.items():
        style = NODE_STYLE.get(node["type"], {"color": "#999", "size": 15, "shape": "dot"})
        net.add_node(
            node_id,
            label=build_label(node),
            title=build_title(node),
            color=style["color"],
            size=style["size"],
            shape=style["shape"],
            font={"size": 10, "color": "#333"},
        )

    for src, rel, tgt, props in edges:
        edge_label = rel
        title_parts = [rel]
        for k, v in props.items():
            if k != "shared_holdings" and v is not None:
                title_parts.append(f"{k}: {v}")
                if k in ("weight", "percentage"):
                    val = v * 100 if isinstance(v, float) and v < 1 else v
                    edge_label = f"{rel} ({val:.1f}%)" if val else rel

        width = 1
        if rel == "OVERLAPS_WITH":
            width = min(props.get("shared_count", 1) * 2, 8)
        elif rel == "HOLDS":
            w = props.get("weight")
            width = max(1, (w or 0) * 20)

        net.add_edge(
            src, tgt,
            label=rel,
            title="<br>".join(title_parts),
            color=EDGE_COLORS.get(rel, "#ccc"),
            width=width,
            arrows="to",
        )

    net.set_options("""
    {
      "physics": {
        "enabled": true,
        "barnesHut": {
          "gravitationalConstant": -3000,
          "centralGravity": 0.3,
          "springLength": 150
        }
      },
      "interaction": {
        "hover": true,
        "tooltipDelay": 100,
        "navigationButtons": true,
        "keyboard": true
      },
      "edges": {
        "font": {"size": 8, "align": "middle"},
        "smooth": {"type": "curvedCW", "roundness": 0.2}
      }
    }
    """)

    net.show(output_file)
    print(f"✓ Saved interactive graph to {output_file}")
    return net

In [ ]:
net = build_pyvis("etf_knowledge_graph.html")

## 8. Filtered views

Generate focused sub-graphs for specific relationship types.

In [ ]:
def build_filtered_pyvis(
    relations: list[str],
    output_file: str,
    title: str = "",
):
    """Build a pyvis graph showing only edges of the given relation types."""
    filtered_edges = [(s, r, t, p) for s, r, t, p in edges if r in relations]
    involved_ids = set()
    for s, _, t, _ in filtered_edges:
        involved_ids.add(s)
        involved_ids.add(t)

    net = Network(
        height="700px",
        width="100%",
        directed=True,
        notebook=True,
        cdn_resources="remote",
        heading=title,
    )
    net.barnes_hut(gravity=-2000, spring_length=120)

    for nid in involved_ids:
        node = nodes[nid]
        style = NODE_STYLE.get(node["type"], {"color": "#999", "size": 15, "shape": "dot"})
        net.add_node(
            nid,
            label=build_label(node),
            title=build_title(node),
            color=style["color"],
            size=style["size"],
            shape=style["shape"],
        )

    for s, r, t, p in filtered_edges:
        width = 1
        if r == "OVERLAPS_WITH":
            width = min(p.get("shared_count", 1) * 2, 8)
        elif r == "HOLDS":
            width = max(1, (p.get("weight") or 0) * 20)

        net.add_edge(s, t, label=r, color=EDGE_COLORS.get(r, "#ccc"), width=width)

    net.show(output_file)
    print(f"✓ {title or output_file}: {len(involved_ids)} nodes, {len(filtered_edges)} edges")
    return net

In [ ]:
build_filtered_pyvis(
    ["HOLDS", "OVERLAPS_WITH"],
    "etf_holdings_overlap.html",
    title="ETF Holdings & Overlap",
)

build_filtered_pyvis(
    ["ISSUED_BY", "TRACKS"],
    "etf_providers_indices.html",
    title="ETF Providers & Indices",
)

build_filtered_pyvis(
    ["EXPOSED_TO", "INVESTS_IN"],
    "etf_geo_sector.html",
    title="ETF Country & Sector Allocation",
)

## 9. Graph statistics & analysis

In [ ]:
G_undirected = G.to_undirected()

print("=== Graph summary ===")
print(f"  Nodes:      {G.number_of_nodes()}")
print(f"  Edges:      {G.number_of_edges()}")
print(f"  Density:    {nx.density(G_undirected):.4f}")
print(f"  Components: {nx.number_connected_components(G_undirected)}")

print("\n=== Most connected nodes (top 15 by degree) ===")
degree_sorted = sorted(G_undirected.degree(), key=lambda x: x[1], reverse=True)[:15]
for nid, deg in degree_sorted:
    ntype = nodes[nid]["type"]
    label = build_label(nodes[nid])
    print(f"  [{ntype:>10}] {label:<50} degree={deg}")

print("\n=== Holdings shared by most ETFs ===")
for h_id, etf_list in sorted(holding_to_etfs.items(), key=lambda x: -len(x[1])):
    if len(etf_list) > 1:
        h_name = nodes[h_id]["props"].get("name", h_id)
        etf_tickers = [nodes[e]["props"].get("ticker", e) for e in etf_list]
        print(f"  {h_name}: held by {', '.join(etf_tickers)}")

## 10. Export tuples to JSON (for later use)

Saves the full graph as a portable JSON file — no dependencies needed to reload it.

In [ ]:
import json

graph_export = {
    "nodes": list(nodes.values()),
    "edges": [
        {"source": s, "relation": r, "target": t, "properties": p}
        for s, r, t, p in edges
    ],
}

with open("etf_kg_data.json", "w") as f:
    json.dump(graph_export, f, indent=2, default=str)

print(f"✓ Exported {len(graph_export['nodes'])} nodes and {len(graph_export['edges'])} edges to etf_kg_data.json")